# TI-748: Causal Impact Analysis — Media Plan Feature

## Executive Summary

**Objective:** Measure the incremental impact of the Media Plan feature on advertiser prospecting KPIs.

**Media Plan** is a beta feature giving advertisers control over network allocation (% of spend per publisher). MNTN auto-recommends allocations; advertisers can accept or customize. Only applies to new prospecting campaigns.

**Methodology:** Per-advertiser Bayesian Structural Time Series (Google's CausalImpact) with platform-wide non-adopter metrics as covariates.

| Design Element | Choice |
|---|---|
| Unit of analysis | Advertiser (all prospecting campaigns aggregated) |
| Intervention date | First `media_plan.create_time` per advertiser |
| Covariates | Platform-wide non-adopter IVR, spend, impressions |
| Aggregation | Weekly |
| Primary metric | IVR (Impression-to-Visit Rate) |
| Minimum data | 6 pre-weeks, 4 post-weeks |

**Key Limitations:**
- Only 26 weeks of BQ agg data available (since 2025-09-01)
- Small N: 10 advertisers with sufficient pre/post data out of 22 adopters
- Short pre-period for early adopters (8 weeks) limits placebo test reliability
- Selection bias: advertisers who chose to use Media Plan may differ systematically

---

## 1. Setup & Configuration

In [ ]:
# Install dependencies (run once)
# !pip install google-cloud-bigquery pycausalimpact pandas numpy matplotlib db-dtypes openpyxl

In [ ]:
import warnings
from datetime import date, timedelta

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from causalimpact import CausalImpact
from google.cloud import bigquery

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

# --- Configuration ---
BQ_PROJECT = "dw-main-silver"
MIN_WEEKLY_IMPRESSIONS = 1000
MAX_IVR = 1.0
MIN_PRE_WEEKS = 6
MIN_POST_WEEKS = 4
PRIMARY_METRIC = "ivr"
COVARIATES = ["platform_ivr", "platform_spend", "platform_impressions"]

METRIC_DEFINITIONS = {
    "ivr":  {"formula": "vv / impressions",   "direction": "higher", "label": "Impression-to-Visit Rate"},
    "cvr":  {"formula": "conversions / vv",    "direction": "higher", "label": "Conversion Rate"},
    "cpa":  {"formula": "spend / conversions", "direction": "lower",  "label": "Cost per Acquisition"},
    "cpv":  {"formula": "spend / vv",          "direction": "lower",  "label": "Cost per Visit"},
    "roas": {"formula": "order_value / spend", "direction": "higher", "label": "Return on Ad Spend"},
}

client = bigquery.Client(project=BQ_PROJECT)
print("Setup complete.")

## 2. Identify Media Plan Adopters

Advertisers with `media_plan_status_id = 3` (active) in `core.media_plan`.
Intervention date = first `create_time` per advertiser.

In [ ]:
adopters = client.query("""
SELECT
    mp.advertiser_id,
    adv.company_name,
    MIN(mp.create_time) AS first_plan_created,
    COUNT(*) AS total_plans,
    COUNT(DISTINCT mp.campaign_group_id) AS campaign_groups_with_plan
FROM `dw-main-silver.core.media_plan` mp
JOIN `dw-main-bronze.integrationprod.advertisers` adv
    ON adv.advertiser_id = mp.advertiser_id
WHERE mp.media_plan_status_id = 3
GROUP BY 1, 2
ORDER BY 3
""").to_dataframe()

adopters["first_plan_created"] = pd.to_datetime(adopters["first_plan_created"])
adopters["intervention_date"] = adopters["first_plan_created"].dt.date

print(f"{len(adopters)} advertisers with active media plans")
adopters[["advertiser_id", "company_name", "intervention_date", "total_plans"]]

## 3. Load Weekly Prospecting KPIs

- Source: `aggregates.agg__daily_sum_by_campaign` joined to `campaigns` (funnel_level=1 for prospecting)
- All adopters use `industry_standard` attribution (confirmed via `r2_advertiser_settings`)
- Weeks with < 1,000 impressions filtered (attribution lag artifacts)

In [ ]:
adopter_list = ",".join(str(x) for x in adopters["advertiser_id"])

weekly_kpis = client.query(f"""
WITH prospecting_campaigns AS (
    SELECT DISTINCT c.campaign_id, c.advertiser_id, c.campaign_group_id
    FROM `dw-main-bronze.integrationprod.campaigns` c
    WHERE c.funnel_level = 1 AND c.deleted = FALSE AND c.is_test = FALSE
)
SELECT
    pc.advertiser_id,
    DATE_TRUNC(a.day, WEEK(MONDAY)) AS week_start,
    CASE WHEN pc.advertiser_id IN ({adopter_list}) THEN TRUE ELSE FALSE END AS is_adopter,
    SUM(a.impressions) AS impressions,
    SUM(a.media_spend + a.data_spend + a.platform_spend) AS spend,
    SUM(a.uniques) AS uniques,
    SUM(a.clicks + a.views + COALESCE(a.competing_views, 0)) AS vv,
    SUM(a.click_conversions + a.view_conversions + COALESCE(a.competing_view_conversions, 0)) AS conversions,
    SUM(a.click_order_value + a.view_order_value + COALESCE(a.competing_view_order_value, 0)) AS order_value
FROM `dw-main-silver.aggregates.agg__daily_sum_by_campaign` a
INNER JOIN prospecting_campaigns pc ON pc.campaign_id = a.campaign_id
WHERE a.day >= '2025-01-01' AND a.impressions > 0
GROUP BY 1, 2, 3
ORDER BY 1, 2
""").to_dataframe()

weekly_kpis["week_start"] = pd.to_datetime(weekly_kpis["week_start"])
for c in ["impressions", "spend", "uniques", "vv", "conversions", "order_value"]:
    weekly_kpis[c] = pd.to_numeric(weekly_kpis[c], errors="coerce").astype(float)

# filter low-impression weeks
weekly_kpis = weekly_kpis[weekly_kpis["impressions"] >= MIN_WEEKLY_IMPRESSIONS].copy()

print(f"{len(weekly_kpis):,} weekly records")
print(f"{weekly_kpis['advertiser_id'].nunique()} total advertisers")
print(f"{weekly_kpis[weekly_kpis['is_adopter']]['advertiser_id'].nunique()} adopters with data")
print(f"Date range: {weekly_kpis['week_start'].min().date()} to {weekly_kpis['week_start'].max().date()}")

## 4. Compute Metrics & Platform Covariates

In [ ]:
def compute_metrics(df):
    """Add derived rate metrics."""
    df = df.copy()
    df["ivr"] = (df["vv"] / df["impressions"].replace(0, np.nan)).clip(upper=MAX_IVR)
    df["cvr"] = (df["conversions"] / df["vv"].replace(0, np.nan)).clip(upper=1.0)
    df["cpa"] = df["spend"] / df["conversions"].replace(0, np.nan)
    df["cpv"] = df["spend"] / df["vv"].replace(0, np.nan)
    df["roas"] = df["order_value"] / df["spend"].replace(0, np.nan)
    return df

# Build platform-wide non-adopter covariates
non_adopter = weekly_kpis[~weekly_kpis["is_adopter"]]
platform = non_adopter.groupby("week_start").agg(
    platform_impressions=("impressions", "sum"),
    platform_spend=("spend", "sum"),
    platform_vv=("vv", "sum"),
).reset_index()
platform["platform_ivr"] = platform["platform_vv"] / platform["platform_impressions"].replace(0, np.nan)

print(f"Platform covariates: {len(platform)} weeks from {non_adopter['advertiser_id'].nunique()} non-adopter advertisers")
platform.tail()

## 5. Run Causal Impact Analysis

For each adopter with sufficient data:
1. Compute weekly metrics for their prospecting campaigns
2. Merge platform covariates
3. Split into pre/post periods based on intervention date
4. Run CausalImpact

In [ ]:
def run_advertiser_ci(advertiser_id, intervention_date, metric, weekly_kpis, platform, covariates):
    """Run CausalImpact for one advertiser + metric. Returns result dict or None."""
    adv = weekly_kpis[weekly_kpis["advertiser_id"] == advertiser_id].copy()
    adv = compute_metrics(adv)
    adv = adv.merge(platform, on="week_start", how="inner")
    adv = adv.set_index("week_start").sort_index()

    intervention_ts = pd.Timestamp(intervention_date)
    intervention_week = intervention_ts - pd.Timedelta(days=intervention_ts.weekday())

    pre = adv[adv.index < intervention_week]
    post = adv[adv.index >= intervention_week]

    if len(pre) < MIN_PRE_WEEKS or len(post) < MIN_POST_WEEKS:
        return None

    pre_period = [pre.index[0], pre.index[-1]]
    post_period = [post.index[0], post.index[-1]]

    ci_data = adv[[metric] + covariates].dropna(subset=[metric])
    ci_data[covariates] = ci_data[covariates].ffill().bfill()
    ci_data = ci_data.astype(float)

    if len(ci_data) < 10:
        return None

    try:
        ci = CausalImpact(ci_data, pre_period, post_period)
        inf = ci.inferences[ci.inferences.index >= post_period[0]]
        actual = ci.post_data.iloc[:, 0].mean()
        predicted = inf["preds"].mean()
        abs_eff = inf["point_effects"].mean()
        rel_eff = abs_eff / predicted if predicted != 0 else np.nan
        post_spend = adv.loc[adv.index >= post_period[0], "spend"].sum()

        return {
            "advertiser_id": advertiser_id,
            "metric": metric,
            "pre_weeks": len(pre), "post_weeks": len(post),
            "actual_avg": actual, "predicted_avg": predicted,
            "absolute_effect": abs_eff, "relative_effect": rel_eff,
            "ci_lower": inf["point_effects_lower"].mean(),
            "ci_upper": inf["point_effects_upper"].mean(),
            "p_value": ci.p_value,
            "significant": ci.p_value < 0.05,
            "post_spend": float(post_spend),
            "ci_object": ci,
        }
    except Exception as e:
        print(f"  Error for {advertiser_id}: {e}")
        return None

In [ ]:
# Run primary metric (IVR) for all adopters
results = []

for _, row in adopters.iterrows():
    adv_id = row["advertiser_id"]
    name = row["company_name"]
    intervention = row["intervention_date"]

    r = run_advertiser_ci(adv_id, intervention, PRIMARY_METRIC, weekly_kpis, platform, COVARIATES)
    if r:
        r["company_name"] = name
        results.append(r)
        sig = "*" if r["significant"] else ""
        print(f"{name:35s} ({adv_id}): {r['relative_effect']:+.2%} (p={r['p_value']:.4f}) {sig}")
    else:
        print(f"{name:35s} ({adv_id}): skipped (insufficient data)")

print(f"\n{len(results)} advertisers analyzed out of {len(adopters)} adopters")

## 6. Results Summary

In [ ]:
def summarize(results, metric):
    """Build and display summary table with aggregate statistics."""
    df = pd.DataFrame([{k: v for k, v in r.items() if k != "ci_object"} for r in results])
    df["effect_pct"] = df["relative_effect"].apply(lambda x: f"{x:+.2%}")
    df["p_fmt"] = df["p_value"].apply(lambda x: f"{x:.4f}")
    df["sig"] = df["significant"].apply(lambda x: "Yes" if x else "No")

    display_cols = ["company_name", "advertiser_id", "pre_weeks", "post_weeks",
                    "actual_avg", "predicted_avg", "effect_pct", "p_fmt", "sig"]
    display(df[display_cols].style.set_caption(f"Causal Impact — {METRIC_DEFINITIONS[metric]['label']}"))

    direction = METRIC_DEFINITIONS[metric]["direction"]
    n = len(df)
    n_sig = df["significant"].sum()
    n_positive = (df["relative_effect"] > 0).sum() if direction == "higher" else (df["relative_effect"] < 0).sum()
    total_spend = df["post_spend"].sum()
    weighted = (df["relative_effect"] * df["post_spend"]).sum() / total_spend if total_spend > 0 else 0

    print(f"\nAdvertisers analyzed:       {n}")
    print(f"Statistically significant:  {n_sig} ({n_sig/n:.0%})")
    print(f"Positive outcome:           {n_positive} ({n_positive/n:.0%})")
    print(f"Mean effect:                {df['relative_effect'].mean():+.2%}")
    print(f"Median effect:              {df['relative_effect'].median():+.2%}")
    print(f"Spend-weighted effect:      {weighted:+.2%}")
    return df

summary_ivr = summarize(results, PRIMARY_METRIC)

## 7. Per-Advertiser Visualizations

In [ ]:
for r in results:
    ci = r["ci_object"]
    print(f"\n{'='*80}")
    print(f"{r['company_name']} ({r['advertiser_id']}) — {PRIMARY_METRIC.upper()}")
    print(f"Effect: {r['relative_effect']:+.2%} (p={r['p_value']:.4f})")
    print(f"{'='*80}")
    ci.plot(figsize=(14, 10))
    plt.show()

## 8. Aggregate Summary Chart

In [ ]:
df_plot = summary_ivr.sort_values("relative_effect")
direction = METRIC_DEFINITIONS[PRIMARY_METRIC]["direction"]

if direction == "higher":
    colors = ["#2ecc71" if (s and e > 0) else "#e74c3c" if (s and e < 0) else "#95a5a6"
              for s, e in zip(df_plot["significant"], df_plot["relative_effect"])]
else:
    colors = ["#2ecc71" if (s and e < 0) else "#e74c3c" if (s and e > 0) else "#95a5a6"
              for s, e in zip(df_plot["significant"], df_plot["relative_effect"])]

fig, ax = plt.subplots(figsize=(12, max(6, len(df_plot) * 0.6)))
bars = ax.barh(df_plot["company_name"], df_plot["relative_effect"] * 100, color=colors)
ax.axvline(x=0, color="black", linewidth=0.8)

for bar, pval in zip(bars, df_plot["p_value"]):
    x = bar.get_width()
    ax.text(x + (1 if x >= 0 else -1), bar.get_y() + bar.get_height() / 2,
            f"p={pval:.3f}", va="center", ha="left" if x >= 0 else "right", fontsize=9)

ax.set_xlabel("Relative Effect (%)")
ax.set_title(f"Media Plan Causal Impact — {PRIMARY_METRIC.upper()} by Advertiser\n"
             f"Green = significant + positive | Red = significant + negative | Gray = not significant")
plt.tight_layout()
plt.show()

## 9. All Metrics Analysis (Optional)

Run CausalImpact across all metrics for a comprehensive view.

In [ ]:
all_metric_results = {}

for metric in METRIC_DEFINITIONS:
    print(f"\n{'='*60}")
    print(f"Metric: {metric.upper()} ({METRIC_DEFINITIONS[metric]['label']})")
    print(f"{'='*60}")

    metric_results = []
    for _, row in adopters.iterrows():
        r = run_advertiser_ci(
            row["advertiser_id"], row["intervention_date"],
            metric, weekly_kpis, platform, COVARIATES
        )
        if r:
            r["company_name"] = row["company_name"]
            metric_results.append(r)

    if metric_results:
        all_metric_results[metric] = summarize(metric_results, metric)
    else:
        print("No advertisers with sufficient data.")

## 10. Cross-Metric Summary

In [ ]:
cross_metric = []
for metric, df in all_metric_results.items():
    info = METRIC_DEFINITIONS[metric]
    n = len(df)
    n_sig = df["significant"].sum()
    n_pos = (df["relative_effect"] > 0).sum() if info["direction"] == "higher" else (df["relative_effect"] < 0).sum()
    total_spend = df["post_spend"].sum()
    weighted = (df["relative_effect"] * df["post_spend"]).sum() / total_spend if total_spend > 0 else 0

    cross_metric.append({
        "Metric": metric.upper(),
        "Label": info["label"],
        "N": n,
        "Significant": f"{n_sig}/{n}",
        "Positive Outcome": f"{n_pos}/{n}",
        "Mean Effect": f"{df['relative_effect'].mean():+.2%}",
        "Median Effect": f"{df['relative_effect'].median():+.2%}",
        "Spend-Weighted": f"{weighted:+.2%}",
    })

cross_df = pd.DataFrame(cross_metric)
display(cross_df.style.set_caption("Cross-Metric Summary — Media Plan Causal Impact"))

## 11. Methodology Notes

### Why CausalImpact with Platform Covariates?

- Each advertiser adopted at a different time → can't pool into one time series
- Platform-wide non-adopter metrics control for seasonality, market trends, and platform-wide changes
- CausalImpact learns the covariate relationship in the pre-period and uses it to predict the counterfactual
- More robust than self-only interrupted time series because external covariates anchor the prediction

### Why NOT Difference-in-Differences?

- Only ~5 eligible-but-non-adopting advertisers as controls → too few
- Parallel trends assumption hard to validate with N=5
- Selection bias: who chose to adopt may differ systematically

### Known Limitations

1. **Short pre-period**: agg data starts 2025-09-01, giving early adopters only ~8 pre-weeks
2. **Small N**: 10 analyzable advertisers out of 22 adopters
3. **Placebo test concerns**: Short pre-periods make placebo tests unreliable (high false positive rate when splitting 8 weeks into 4/4)
4. **Selection bias**: Advertisers who adopted may be more engaged, tech-savvy, or already improving
5. **Confounders**: Other changes (budget shifts, seasonality, new creatives) during the post-period
6. **Attribution**: All adopters use industry_standard attribution — competing views included